In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BikeStoreETL") \
    .enableHiveSupport() \
    .getOrCreate()

In [27]:
spark.conf.set("spark.sql.shuffle.partitions", 10)

In [2]:
orders = spark.read.csv("/staging/orders", inferSchema=True)
order_items = spark.read.csv("/staging/order_items", inferSchema=True)
products = spark.read.csv("/staging/products", inferSchema=True)
stores = spark.read.csv("/staging/stores", inferSchema=True)
customers = spark.read.csv("/staging/customers", inferSchema=True)

In [3]:
orders.show(5)

+---+----+---+----------+----------+----------+---+---+
|_c0| _c1|_c2|       _c3|       _c4|       _c5|_c6|_c7|
+---+----+---+----------+----------+----------+---+---+
|  1| 259|  4|2016-01-01|2016-01-03|2016-01-03|  1|  2|
|  2|1212|  4|2016-01-01|2016-01-04|2016-01-03|  2|  6|
|  3| 523|  4|2016-01-02|2016-01-05|2016-01-03|  2|  7|
|  4| 175|  4|2016-01-03|2016-01-04|2016-01-05|  1|  3|
|  5|1324|  4|2016-01-03|2016-01-06|2016-01-06|  2|  6|
+---+----+---+----------+----------+----------+---+---+
only showing top 5 rows



In [4]:
orders.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)



In [5]:
order_items.show(5)

+---+---+---+---+-------+----+
|_c0|_c1|_c2|_c3|    _c4| _c5|
+---+---+---+---+-------+----+
|  1|  1| 20|  1| 599.99| 0.2|
|  1|  2|  8|  2|1799.99|0.07|
|  1|  3| 10|  2| 1549.0|0.05|
|  1|  4| 16|  2| 599.99|0.05|
|  1|  5|  4|  1|2899.99| 0.2|
+---+---+---+---+-------+----+
only showing top 5 rows



In [7]:
products.show(5)

+---+--------------------+---+---+----+-------+
|_c0|                 _c1|_c2|_c3| _c4|    _c5|
+---+--------------------+---+---+----+-------+
|  1|     Trek 820 - 2016|  9|  6|2016| 379.99|
|  2|Ritchey Timberwol...|  5|  6|2016| 749.99|
|  3|Surly Wednesday F...|  8|  6|2016| 999.99|
|  4|Trek Fuel EX 8 29...|  9|  6|2016|2899.99|
|  5|Heller Shagamaw F...|  3|  6|2016|1320.99|
+---+--------------------+---+---+----+-------+
only showing top 5 rows



In [6]:
def read_and_rename(path, columns):
    return spark.read.csv(path, header=False, inferSchema=True).toDF(*columns)

In [7]:
df_customers = read_and_rename("/staging/customers", [
    "customer_id",
    "first_name",
    "last_name",
    "phone",
    "email",
    "street",
    "city",
    "state",
    "zip_code"
])

In [8]:
df_orders = read_and_rename("/staging/orders", [
    "order_id",
    "customer_id",
    "order_status",
    "order_date",
    "required_date",
    "shipped_date",
    "store_id",
    "staff_id"
])

In [9]:
df_order_items = read_and_rename("/staging/order_items", [
    "order_id",
    "item_id",
    "product_id",
    "quantity",
    "list_price",
    "discount"
])

In [10]:
df_products = read_and_rename("/staging/products", [
    "product_id",
    "product_name",
    "brand_id",
    "category_id",
    "model_year",
    "list_price"
])

In [11]:
df_brands = read_and_rename("/staging/brands", [
    "brand_id",
    "brand_name"
])

In [12]:
df_categories = read_and_rename("/staging/categories", [
    "category_id",
    "category_name"
])

In [13]:
df_stores = read_and_rename("/staging/stores", [
    "store_id",
    "store_name",
    "phone",
    "email",
    "street",
    "city",
    "state",
    "zip_code"
])

In [14]:
df_staffs = read_and_rename("/staging/staffs", [
    "staff_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "active",
    "store_id",
    "manager_id"
])

In [15]:
df_stocks = read_and_rename("/staging/stocks", [
    "store_id",
    "product_id",
    "quantity"
])

In [16]:
df_orders.printSchema()
df_orders.show(5)

df_order_items.printSchema()
df_customers.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- required_date: string (nullable = true)
 |-- shipped_date: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)

+--------+-----------+------------+----------+-------------+------------+--------+--------+
|order_id|customer_id|order_status|order_date|required_date|shipped_date|store_id|staff_id|
+--------+-----------+------------+----------+-------------+------------+--------+--------+
|       1|        259|           4|2016-01-01|   2016-01-03|  2016-01-03|       1|       2|
|       2|       1212|           4|2016-01-01|   2016-01-04|  2016-01-03|       2|       6|
|       3|        523|           4|2016-01-02|   2016-01-05|  2016-01-03|       2|       7|
|       4|        175|           4|2016-01-03|   2016-01-04|  2016-01-05|       1|       3|
|      

In [17]:
from pyspark.sql.functions import col

df_orders = df_orders \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("required_date", col("required_date").cast("date")) \
    .withColumn("shipped_date", col("shipped_date").cast("date"))

In [18]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)



In [19]:
from pyspark.sql.functions import expr

df_order_items = df_order_items.withColumn(
    "total_price",
    expr("quantity * list_price * (1 - discount)")
)

In [20]:
df_order_items.select("quantity", "list_price", "discount", "total_price").show(5)

+--------+----------+--------+------------------+
|quantity|list_price|discount|       total_price|
+--------+----------+--------+------------------+
|       1|    599.99|     0.2|           479.992|
|       2|   1799.99|    0.07|3347.9813999999997|
|       2|    1549.0|    0.05|            2943.1|
|       2|    599.99|    0.05|          1139.981|
|       1|   2899.99|     0.2|2319.9919999999997|
+--------+----------+--------+------------------+
only showing top 5 rows



In [21]:
df_sales = df_order_items \
    .join(df_orders, "order_id") \
    .join(df_customers, "customer_id")

In [22]:
df_sales.show(5)
df_sales.printSchema()

+-----------+--------+-------+----------+--------+----------+--------+------------------+------------+----------+-------------+------------+--------+--------+----------+---------+-----+--------------------+--------------------+----------+-----+--------+
|customer_id|order_id|item_id|product_id|quantity|list_price|discount|       total_price|order_status|order_date|required_date|shipped_date|store_id|staff_id|first_name|last_name|phone|               email|              street|      city|state|zip_code|
+-----------+--------+-------+----------+--------+----------+--------+------------------+------------+----------+-------------+------------+--------+--------+----------+---------+-----+--------------------+--------------------+----------+-----+--------+
|        259|       1|      1|        20|       1|    599.99|     0.2|           479.992|           4|2016-01-01|   2016-01-03|  2016-01-03|       1|       2| Johnathan|Velazquez| null|johnathan.velazqu...|9680 E. Somerset ...|Pleasanton|

In [23]:
from pyspark.sql.functions import sum, count

kpi_store = df_sales.groupBy("store_id").agg(
    sum("total_price").alias("total_sales"),
    count("order_id").alias("total_orders")
)

In [24]:
kpi_products = df_sales.groupBy("product_id").agg(
    sum("total_price").alias("revenue")
).orderBy("revenue", ascending=False)

In [25]:
kpi_customers = df_sales.groupBy("customer_id").agg(
    sum("total_price").alias("spending")
).orderBy("spending", ascending=False)

In [26]:
kpi_store.show()
kpi_products.show(5)
kpi_customers.show(5)

+--------+------------------+------------+
|store_id|       total_sales|total_orders|
+--------+------------------+------------+
|       1|1605823.0364999974|        1006|
|       3| 867542.2436000014|         521|
|       2| 5215751.277499983|        3195|
+--------+------------------+------------+



+----------+------------------+
|product_id|           revenue|
+----------+------------------+
|         7|       555558.6111|
|         9| 389248.7025000002|
|         4|368472.72939999995|
|        11| 226765.5509999999|
|        56|211584.61529999998|
+----------+------------------+
only showing top 5 rows



+-----------+------------------+
|customer_id|          spending|
+-----------+------------------+
|         94| 34807.93919999999|
|         10|        33634.2604|
|         75|        32803.0062|
|          6|32675.072499999995|
|         16|31925.885700000003|
+-----------+------------------+
only showing top 5 rows



In [29]:
spark.sql("CREATE DATABASE bike_dw")

2026-04-08 20:18:44,886 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-04-08 20:18:44,924 WARN metastore.ObjectStore: Failed to get database bike_dw, returning NoSuchObjectException


DataFrame[]

In [30]:
spark.sql("USE bike_dw")

DataFrame[]

In [31]:
kpi_store.write.mode("overwrite").saveAsTable("bike_dw.sales_kpi")

2026-04-08 20:19:04,022 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-04-08 20:19:04,141 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-04-08 20:19:04,141 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-04-08 20:19:04,141 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [32]:
spark.sql("SELECT * FROM bike_dw.sales_kpi").show()

+--------+------------------+------------+
|store_id|       total_sales|total_orders|
+--------+------------------+------------+
|       3| 867542.2436000014|         521|
|       1|1605823.0364999974|        1006|
|       2| 5215751.277499983|        3195|
+--------+------------------+------------+



In [33]:
spark.sql("""
CREATE EXTERNAL TABLE bike_dw.sales_kpi_ext (
    store_id INT,
    total_sales DOUBLE,
    total_orders INT
)
STORED AS PARQUET
LOCATION '/user/hive/warehouse/bike_dw.db/sales_kpi'
""")

DataFrame[]

In [34]:
spark.sql("SELECT * FROM bike_dw.sales_kpi_ext").show()

2026-04-08 20:21:01,268 ERROR executor.Executor: Exception in task 0.0 in stage 63.0 (TID 661)
org.apache.spark.sql.execution.QueryExecutionException: Parquet column cannot be converted in file hdfs://localhost:9000/user/hive/warehouse/bike_dw.db/sales_kpi/part-00001-09b45814-2c35-4626-b077-a824acdf4cf7-c000.snappy.parquet. Column: [total_orders], Expected: int, Found: INT64
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:179)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:93)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:503)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNex

Py4JJavaError: An error occurred while calling o244.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 63.0 failed 1 times, most recent failure: Lost task 0.0 in stage 63.0 (TID 661) (192.168.91.132 executor driver): org.apache.spark.sql.execution.QueryExecutionException: Parquet column cannot be converted in file hdfs://localhost:9000/user/hive/warehouse/bike_dw.db/sales_kpi/part-00001-09b45814-2c35-4626-b077-a824acdf4cf7-c000.snappy.parquet. Column: [total_orders], Expected: int, Found: INT64
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:179)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:93)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:503)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.s

In [35]:
spark.sql("DROP TABLE bike_dw.sales_kpi_ext")

DataFrame[]

In [36]:
spark.sql("""
CREATE EXTERNAL TABLE bike_dw.sales_kpi_ext (
    store_id INT,
    total_sales DOUBLE,
    total_orders BIGINT
)
STORED AS PARQUET
LOCATION '/user/hive/warehouse/bike_dw.db/sales_kpi'
""")

DataFrame[]

In [37]:
spark.sql("SELECT * FROM bike_dw.sales_kpi_ext").show()

+--------+------------------+------------+
|store_id|       total_sales|total_orders|
+--------+------------------+------------+
|       3| 867542.2436000014|         521|
|       1|1605823.0364999974|        1006|
|       2| 5215751.277499983|        3195|
+--------+------------------+------------+



In [38]:
kpi_products.write.mode("overwrite").saveAsTable("bike_dw.product_kpi")
kpi_customers.write.mode("overwrite").saveAsTable("bike_dw.customer_kpi")

In [39]:
spark.sql("SELECT * FROM bike_dw.product_kpi").show(5)
spark.sql("SELECT * FROM bike_dw.customer_kpi").show(5)

+----------+------------------+
|product_id|           revenue|
+----------+------------------+
|        15|65061.572400000034|
|        59| 59513.77109999999|
|        48| 57359.61759999999|
|       155|        54359.9547|
|        18|54270.629999999976|
+----------+------------------+
only showing top 5 rows

+-----------+-----------------+
|customer_id|         spending|
+-----------+-----------------+
|        870|        8309.4449|
|       1084|        8303.3719|
|       1192|         8299.965|
|       1069|         8280.543|
|        181|8273.955899999999|
+-----------+-----------------+
only showing top 5 rows



In [40]:
kpi_products.printSchema()
kpi_customers.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- revenue: double (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- spending: double (nullable = true)



In [41]:
spark.sql("""
CREATE EXTERNAL TABLE bike_dw.product_kpi_ext (
    product_id INT,
    revenue DOUBLE
)
STORED AS PARQUET
LOCATION '/user/hive/warehouse/bike_dw.db/product_kpi'
""")

DataFrame[]

In [42]:
spark.sql("""
CREATE EXTERNAL TABLE bike_dw.customer_kpi_ext (
    customer_id INT,
    spending DOUBLE
)
STORED AS PARQUET
LOCATION '/user/hive/warehouse/bike_dw.db/customer_kpi'
""")

DataFrame[]

In [43]:
spark.sql("SELECT * FROM bike_dw.product_kpi_ext").show(5)
spark.sql("SELECT * FROM bike_dw.customer_kpi_ext").show(5)

+----------+------------------+
|product_id|           revenue|
+----------+------------------+
|        15|65061.572400000034|
|        59| 59513.77109999999|
|        48| 57359.61759999999|
|       155|        54359.9547|
|        18|54270.629999999976|
+----------+------------------+
only showing top 5 rows

+-----------+-----------------+
|customer_id|         spending|
+-----------+-----------------+
|        870|        8309.4449|
|       1084|        8303.3719|
|       1192|         8299.965|
|       1069|         8280.543|
|        181|8273.955899999999|
+-----------+-----------------+
only showing top 5 rows



In [44]:
spark.sql("""
CREATE EXTERNAL TABLE bike_dw.hbase_sales (
    key STRING,
    total_sales DOUBLE,
    total_orders BIGINT
)
STORED BY 'org.apache.hadoop.hive.hbase.HBaseStorageHandler'
WITH SERDEPROPERTIES (
    "hbase.columns.mapping" = ":key,cf:total_sales,cf:total_orders"
)
TBLPROPERTIES ("hbase.table.name" = "sales_hbase")
""")

ParseException: 
Operation not allowed: STORED BY(line 7, pos 0)

== SQL ==

CREATE EXTERNAL TABLE bike_dw.hbase_sales (
    key STRING,
    total_sales DOUBLE,
    total_orders BIGINT
)
STORED BY 'org.apache.hadoop.hive.hbase.HBaseStorageHandler'
^^^
WITH SERDEPROPERTIES (
    "hbase.columns.mapping" = ":key,cf:total_sales,cf:total_orders"
)
TBLPROPERTIES ("hbase.table.name" = "sales_hbase")
